# Practice 2 — Evaluation protocol (both challenges)

**Role:** R4 — Evaluation and validation officer

In [1]:
# Frozen project-wide random state (course rule): used only where an API consumes randomness
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


## 0. Setup

In [2]:
# Libraries: pandas/numpy for data, and scikit-learn for splitters, preprocessing and KNN models
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (GroupKFold, KFold, StratifiedGroupKFold,
                                     StratifiedKFold, cross_validate)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

## 1. Data

`row_id`, `feature_group` and `config_group` are loaded but are **never** estimator inputs.

In [3]:
# Load the development tables and the group metadata (groups only define the folds)
cyber = pd.read_csv("../datos/cyber/cyber_development.csv")
cyber_groups = pd.read_csv("../datos/cyber/cyber_groups.csv")
storage = pd.read_csv("../datos/storage/storage_development.csv")
storage_groups = pd.read_csv("../datos/storage/storage_groups.csv")

# Column roles: nominal columns are one-hot encoded, numeric columns are scaled
CYBER_NOMINAL = ["proto", "service", "conn_state"]
CYBER_NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
                 "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
STORAGE_NOMINAL = ["load_type", "io_type", "raid", "device_type"]  # raid is categorical ("4+2")
STORAGE_NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]

# Challenge A: features X, target y (attack_code) and grouping key g (feature_group)
X_cyber = cyber[CYBER_NOMINAL + CYBER_NUMERIC]
y_cyber = cyber["attack_code"]
g_cyber = cyber_groups["feature_group"]

# Challenge B: features X, modelling target z and grouping key g (config_group)
X_storage = storage[STORAGE_NOMINAL + STORAGE_NUMERIC]
z_storage = np.log10(storage["latency_ns"])   # modelling target z = log10(latency_ns)
g_storage = storage_groups["config_group"]

print(X_cyber.shape, X_storage.shape)

(34928, 11) (12190, 9)


## 2. Course-fixed comparison pipelines

Fixed by the statement for P2.2 (KNN with `n_neighbors=5`).

In [4]:
# Challenge A pipeline: one-hot on nominal columns; log1p then standardisation on numeric
# columns; KNN classifier. Being inside a Pipeline, every transformer is fitted on the
# training portion of each fold only (no leakage from the validation portion).
cyber_pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("nominal", OneHotEncoder(handle_unknown="ignore"), CYBER_NOMINAL),
        ("numeric", Pipeline([("log1p", FunctionTransformer(np.log1p)),
                               ("scale", StandardScaler())]), CYBER_NUMERIC),
    ])),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

# Challenge B pipeline: one-hot on nominal columns; standardisation on numeric columns;
# KNN regressor (it predicts z = log10(latency_ns))
storage_pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("nominal", OneHotEncoder(handle_unknown="ignore"), STORAGE_NOMINAL),
        ("numeric", StandardScaler(), STORAGE_NUMERIC),
    ])),
    ("knn", KNeighborsRegressor(n_neighbors=5)),
])

## 3. P2.2 (a) — Build the grouped protocols

- Cyber: `StratifiedGroupKFold(n_splits=5)`, groups = `feature_group`.
- Storage: `GroupKFold(n_splits=5)`, groups = `config_group`.

Pass `PROJECT_RANDOM_STATE` only where randomness is actually used (e.g. `shuffle=True`).

In [5]:
# Sanity check: the group tables must be row-aligned with the development tables,
# because groups are used by position below
assert (cyber["row_id"] == cyber_groups["row_id"]).all()
assert (storage["row_id"] == storage_groups["row_id"]).all()

# Our own grouped splitters. shuffle=True makes them consume randomness, so the frozen
# seed is passed
cv_cyber = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=PROJECT_RANDOM_STATE)
cv_storage = GroupKFold(n_splits=5, shuffle=True, random_state=PROJECT_RANDOM_STATE)

# Each fold is stored as (train_idx, val_idx): row positions of the two portions
folds_cyber = list(cv_cyber.split(X_cyber, y_cyber, groups=g_cyber))
folds_storage = list(cv_storage.split(X_storage, z_storage, groups=g_storage))

# Report the (training rows, validation rows) of every fold
for name, folds in [("cyber", folds_cyber), ("storage", folds_storage)]:
    print(name, [(len(tr), len(va)) for tr, va in folds])

cyber [(27942, 6986), (27942, 6986), (27943, 6985), (27942, 6986), (27943, 6985)]
storage [(9750, 2440), (9750, 2440), (9750, 2440), (9755, 2435), (9755, 2435)]


## 4. P2.2 (b) — Verify and report the invariants

For every fold, check and **print**:
- cyber: no `feature_group` appears in both the training and the validation part;
- storage: no `config_group` is split across folds;
- cyber: every class is present in every validation part (asked in the memo).

Then explain how this differs from ordinary row-wise cross-validation.

In [6]:
n_classes = y_cyber.nunique()

# Cyber invariants, per fold: no feature_group on both sides, all classes present in the
# validation part (including the rare ransomware class, code 7). assert stops on a violation.
print("Cyber: StratifiedGroupKFold on feature_group")
for k, (tr, va) in enumerate(folds_cyber, start=1):
    shared = set(g_cyber.iloc[tr]) & set(g_cyber.iloc[va])   # must be empty
    classes_val = y_cyber.iloc[va].nunique()
    print(f"  fold {k}: shared feature_group = {len(shared)} | "
          f"classes in validation = {classes_val}/{n_classes} | "
          f"ransomware (7) in validation = {(y_cyber.iloc[va] == 7).sum()}")
    assert not shared

# Storage invariants, per fold: no config_group on both sides, and every configuration
# is validated in exactly one fold (all five replicates stay together)
print("Storage: GroupKFold on config_group")
seen = {}   # config_group -> fold where it was validated
for k, (tr, va) in enumerate(folds_storage, start=1):
    shared = set(g_storage.iloc[tr]) & set(g_storage.iloc[va])
    print(f"  fold {k}: shared config_group = {len(shared)} | "
          f"configurations in validation = {g_storage.iloc[va].nunique()}")
    assert not shared
    for g in g_storage.iloc[va].unique():
        assert g not in seen, "config_group appears in the validation part of two folds"
        seen[g] = k
print("  every config_group is validated in exactly one fold:",
      len(seen) == g_storage.nunique())

Cyber: StratifiedGroupKFold on feature_group
  fold 1: shared feature_group = 0 | classes in validation = 10/10 | ransomware (7) in validation = 42
  fold 2: shared feature_group = 0 | classes in validation = 10/10 | ransomware (7) in validation = 42
  fold 3: shared feature_group = 0 | classes in validation = 10/10 | ransomware (7) in validation = 41
  fold 4: shared feature_group = 0 | classes in validation = 10/10 | ransomware (7) in validation = 41
  fold 5: shared feature_group = 0 | classes in validation = 10/10 | ransomware (7) in validation = 42
Storage: GroupKFold on config_group
  fold 1: shared config_group = 0 | configurations in validation = 488
  fold 2: shared config_group = 0 | configurations in validation = 488
  fold 3: shared config_group = 0 | configurations in validation = 488
  fold 4: shared config_group = 0 | configurations in validation = 487
  fold 5: shared config_group = 0 | configurations in validation = 487
  every config_group is validated in exactly one 

_Interpretation:_

All the checks pass in the five folds of both challenges.

- **Cyber:** no `feature_group` appears in both the training part and the validation part, and all 10 classes appear in every validation part. `ransomware` (class 7) has only 41–42 rows per validation part (208 in total).
- **Storage:** no `config_group` is split across folds, and every configuration is in the validation part of exactly one fold (about 488 configurations per fold).

**Difference with ordinary row-wise cross-validation.** In ordinary K-fold each *row* is assigned to a fold on its own, so rows of the same group can end up in both the training part and the validation part. Here whole *groups* go to one fold, so the validation part only has groups that the model did not see in training. This matters most in storage, where the 5 replicates of a configuration are almost identical. The price is that the stratification is only approximate, because groups must stay whole. That is why we check that every class, especially `ransomware`, reaches every validation part.

## 5. P2.2 (c) — Storage: grouped vs row-wise

Same `storage_pipeline`, metric MAE(z); **only the splitting protocol changes**: `GroupKFold` vs ordinary `KFold`. Row-wise is known to be optimistic here — reproduce it and explain the mechanism (replicates of one configuration on both sides).

In [7]:
# Compare two split protocols with the SAME pipeline and metric (MAE of z); only the
# splitting changes. The grouped protocol reuses the folds built above.
# Replicates of one configuration are contiguous in the file, so the row-wise KFold must
# shuffle; otherwise most replicates would stay together by accident of row order.
protocols_storage = {
    "grouped GroupKFold": folds_storage,
    "row-wise KFold": KFold(n_splits=5, shuffle=True, random_state=PROJECT_RANDOM_STATE),
}

# Cross-validate under each protocol and collect the per-fold MAE(z), its mean and spread
rows = []
for name, cv in protocols_storage.items():
    res = cross_validate(storage_pipeline, X_storage, z_storage, cv=cv,
                         groups=g_storage if name.startswith("grouped") else None,
                         scoring="neg_mean_absolute_error")
    mae = -res["test_score"]   # scikit-learn returns the negated error
    rows.append({"protocol": name, **{f"fold {i+1}": m for i, m in enumerate(mae)},
                 "mean MAE(z)": mae.mean(), "spread (max-min)": mae.max() - mae.min()})

storage_comparison = pd.DataFrame(rows).set_index("protocol")
display(storage_comparison.round(4))

# Observed gap between the protocols, as a difference and as a ratio
diff = (storage_comparison.loc["grouped GroupKFold", "mean MAE(z)"]
        - storage_comparison.loc["row-wise KFold", "mean MAE(z)"])
ratio = (storage_comparison.loc["grouped GroupKFold", "mean MAE(z)"]
         / storage_comparison.loc["row-wise KFold", "mean MAE(z)"])
print(f"grouped - row-wise = {diff:.4f} in MAE(z)  (grouped error is {ratio:.2f}x the row-wise one)")

,fold 1,fold 2,fold 3,fold 4,fold 5,mean MAE(z),spread (max-min)
protocol,,,,,,,
grouped GroupKFold,0.2036,0.2110,0.2063,0.2221,0.2189,0.2124,0.0185
row-wise KFold,0.0804,0.0794,0.0836,0.0858,0.0755,0.0809,0.0104


grouped - row-wise = 0.1315 in MAE(z)  (grouped error is 2.62x the row-wise one)


_Interpretation:_

We use the same five-neighbour KNN pipeline and MAE(z); only the split changes.

- GroupKFold mean MAE(z): about 0.2124.
- Row-wise KFold mean MAE(z): about 0.0809.
- Grouped minus row-wise: about 0.1315; the grouped error is 2.62 times larger.

The row-wise estimate is optimistic for generalisation to new configurations. Each configuration has five rows with exactly the same nine feature values. A row-wise split can put replicates in training and validation, allowing zero-distance training neighbours from an already seen configuration. GroupKFold keeps the replicates together, so validation configurations are unseen. The two protocols validate different rows in folds with the same number; these are not paired model comparisons or a significance test.

If exponentiated, the log errors correspond to multiplicative factors of about 1.63 and 1.20. These are not errors in ns, prediction intervals or guaranteed coverage bounds.

Replicates are contiguous in the file, so shuffling the row-wise KFold exposes the intended row-wise mixing. No experiment with alternative seeds or an unshuffled comparator is reported.

## 6. P2.2 (d) — Cyber: grouped vs ungrouped

Same `cyber_pipeline`, metric macro F1; `StratifiedGroupKFold` vs `StratifiedKFold`. Report the difference **as measured**, whatever its size and sign.

In [8]:
# Compare two split protocols with the SAME pipeline and metric (macro F1); only the
# splitting changes: our grouped folds vs an ordinary stratified split that ignores groups
protocols_cyber = {
    "grouped StratifiedGroupKFold": folds_cyber,
    "ungrouped StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True,
                                                 random_state=PROJECT_RANDOM_STATE),
}

# Cross-validate under each protocol and collect the per-fold macro F1, its mean and spread
rows = []
for name, cv in protocols_cyber.items():
    res = cross_validate(cyber_pipeline, X_cyber, y_cyber, cv=cv, scoring="f1_macro")
    f1 = res["test_score"]
    rows.append({"protocol": name, **{f"fold {i+1}": s for i, s in enumerate(f1)},
                 "mean macro F1": f1.mean(), "spread (max-min)": f1.max() - f1.min()})

cyber_comparison = pd.DataFrame(rows).set_index("protocol")
display(cyber_comparison.round(4))

# Observed difference, reported with its sign whatever its size
diff = (cyber_comparison.loc["grouped StratifiedGroupKFold", "mean macro F1"]
        - cyber_comparison.loc["ungrouped StratifiedKFold", "mean macro F1"])
print(f"grouped - ungrouped = {diff:+.4f} in macro F1")

,fold 1,fold 2,fold 3,fold 4,fold 5,mean macro F1,spread (max-min)
protocol,,,,,,,
grouped StratifiedGroupKFold,0.9419,0.9466,0.9516,0.9399,0.9408,0.9442,0.0117
ungrouped StratifiedKFold,0.9372,0.9404,0.9476,0.9536,0.9424,0.9443,0.0164


grouped - ungrouped = -0.0001 in macro F1


_Interpretation:_

We use the same five-neighbour KNN pipeline and macro F1; only the split changes.

- StratifiedGroupKFold mean macro F1: 0.9442.
- StratifiedKFold mean macro F1: 0.9443.
- Grouped minus ungrouped: approximately -0.0001.

The observed difference between means is small relative to each protocol's descriptive fold range (about 0.012 and 0.016). This is not proof of equivalence or of no effect. A fold number under one protocol does not identify the same validation rows under the other, so these folds are not paired model comparisons.

There are 34,883 groups for 34,928 rows. The 38 multirow groups contain 83 rows, including 45 rows beyond one representative per group. Their identical predictors have conflicting labels, as the data exploration shows. This small amount of duplication provides context for the small gap but does not establish its cause. An ungrouped estimate can be higher, lower or nearly equal: the direction is an empirical result, not a consequence of grouping.

Grouping remains necessary because the intended estimate is generalisation to unseen exact feature groups.

## 7. P2.2 (e) — Why splitting identical feature groups is undesirable

Argue in terms of **what the validation score is meant to estimate**, not the size of the gap.

_Argument:_

1. **What is estimated.** The validation score should describe prediction on exact feature groups not present in training, matching the course's hidden split.
2. **What a split group changes.** An identical training vector can become a zero-distance neighbour of a validation row. That evaluates partly seen predictor groups, even if their labels conflict; it does not guarantee a correct prediction or a score increase.
3. **Why the observed sign is not decisive.** In cyber the ungrouped score may be higher, lower or similar. The size and direction depend on the data and pipeline. A small gap does not validate the ungrouped estimand.
4. **Why selection must use the right protocol.** Selecting neighbours or model families against the wrong generalisation unit optimises a different question from the project's question.

The grouped protocol is therefore justified by the population it represents, not by a presumed inflation or an informal significance claim.

## 8. P2.2 (f) — Freeze our construction; compare with the canonical folds

The course-owned `cyber_folds.csv` and `storage_folds.csv` (columns `row_id`, `fold_id`) are stored in `datos/cyber/` and `datos/storage/`. Here we load them, verify their invariants, and compare them with our own assignment. Our construction is frozen from this point on.

In [9]:
# Load the course-owned canonical fold assignment (one fold_id per row)
canonical_cyber = pd.read_csv("../datos/cyber/cyber_folds.csv")      # row_id, fold_id
canonical_storage = pd.read_csv("../datos/storage/storage_folds.csv")  # row_id, fold_id

# The canonical files must be row-aligned with the development tables
assert (canonical_cyber["row_id"] == cyber["row_id"]).all()
assert (canonical_storage["row_id"] == storage["row_id"]).all()


def to_fold_id(folds, n_rows):
    """Turn our list of (train_idx, val_idx) into a fold_id vector."""
    fold_id = np.full(n_rows, -1)
    for k, (_, val_idx) in enumerate(folds):
        fold_id[val_idx] = k
    return fold_id


# Put our assignment and the canonical one in the same format (fold_id per row)
ours = {"cyber": to_fold_id(folds_cyber, len(cyber)),
        "storage": to_fold_id(folds_storage, len(storage))}
canonical = {"cyber": canonical_cyber["fold_id"].to_numpy(),
             "storage": canonical_storage["fold_id"].to_numpy()}
groups = {"cyber": g_cyber, "storage": g_storage}

# 1) Check the invariants on the canonical assignment: fold sizes, and no group spread
#    over more than one fold
for name in ["cyber", "storage"]:
    fid = pd.Series(canonical[name])
    folds_per_group = fid.groupby(groups[name].to_numpy()).nunique()
    print(f"{name}: fold ids = {sorted(fid.unique())}, rows per fold = "
          f"{fid.value_counts().sort_index().tolist()}, "
          f"groups appearing in more than one fold = {(folds_per_group > 1).sum()}")
# Rare class: ransomware (code 7) rows in each canonical fold
print("cyber ransomware rows per canonical fold:",
      pd.Series(canonical["cyber"])[y_cyber.to_numpy() == 7].value_counts().sort_index().tolist())

# 2) Compare our assignment with the canonical one. If every row of the table has a single
#    non-zero cell (and every column too), both define the same partition of the rows
#    up to a relabelling of the folds.
for name in ["cyber", "storage"]:
    table = pd.crosstab(ours[name], canonical[name],
                        rownames=["our fold"], colnames=["canonical fold"])
    identical = (table.gt(0).sum(axis=1) == 1).all() and (table.gt(0).sum(axis=0) == 1).all()
    print(f"\n{name}: same partition of the rows (up to fold relabelling)? {identical}")
    display(table)

cyber: fold ids = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)], rows per fold = [6986, 6986, 6985, 6986, 6985], groups appearing in more than one fold = 0
storage: fold ids = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)], rows per fold = [2440, 2440, 2440, 2435, 2435], groups appearing in more than one fold = 0
cyber ransomware rows per canonical fold: [42, 42, 41, 41, 42]

cyber: same partition of the rows (up to fold relabelling)? True


canonical fold,0,1,2,3,4
our fold,,,,,
0,6986,0,0,0,0
1,0,6986,0,0,0
2,0,0,6985,0,0
3,0,0,0,6986,0
4,0,0,0,0,6985



storage: same partition of the rows (up to fold relabelling)? False


canonical fold,0,1,2,3,4
our fold,,,,,
0,445,510,535,425,525
1,455,445,510,560,470
2,510,485,455,520,470
3,530,565,425,450,465
4,500,435,515,480,505


_Discrepancy:_

**Challenge A: same folds.** Our assignment is the same as the canonical one. Each of our folds contains exactly the same rows as the canonical fold with the same number (sizes 6,986 / 6,986 / 6,985 / 6,986 / 6,985). The canonical file also keeps every `feature_group` in a single fold and has 42 / 42 / 41 / 41 / 42 `ransomware` rows in the validation parts. So our construction, `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2627)`, reproduces it.

**Challenge B: different folds.** Our assignment is not the same. Each of our folds is spread over all five canonical folds (roughly 425–565 rows from each). The fold sizes are the same (2,440 / 2,440 / 2,440 / 2,435 / 2,435), and the canonical file also keeps every `config_group` in a single fold. So both are valid grouped splits that only differ in *which* configurations go to which fold. Our configured GroupKFold and the course sidecar define different valid partitions. We adopt the course membership rather than trying to reconstruct it or searching seeds.

**Consequence.** Our own folds stay in the notebook to show that we understand how they are built. From here on, every scored result uses the canonical folds.

## 9. P2.2 (g) — Adopt the canonical assignment

From here on, every scored result uses the canonical folds.

In [10]:
from sklearn.model_selection import PredefinedSplit


def canonical_cv(challenge):
    """Canonical 5-fold cv for cross_validate: 'cyber' or 'storage'.

    Reads the course-owned fold_id file; fold_id is never an estimator input.
    """
    path = f"../datos/{challenge}/{challenge}_folds.csv"
    return PredefinedSplit(test_fold=pd.read_csv(path)["fold_id"].to_numpy())


# The two canonical cv objects used by every scored result from here on
cv_cyber_canonical = canonical_cv("cyber")
cv_storage_canonical = canonical_cv("storage")

# Check: 5 folds, and each validation row belongs to exactly one validation portion
for name, cv, n in [("cyber", cv_cyber_canonical, len(cyber)),
                    ("storage", cv_storage_canonical, len(storage))]:
    val_rows = np.concatenate([va for _, va in cv.split()])
    print(f"{name}: {cv.get_n_splits()} folds, "
          f"validation partitions all {n} rows once: {len(val_rows) == n == len(set(val_rows))}")

cyber: 5 folds, validation partitions all 34928 rows once: True
storage: 5 folds, validation partitions all 12190 rows once: True


### Common evaluation functions


`evaluate_cyber` and `evaluate_storage` take any estimator (or Pipeline), the features, the target and a canonical `cv`, and return the metric sets of the metric contract. They refit a clone on every fold, so preprocessing inside the Pipeline is fitted on the training portion only.

- **Challenge A:** per-fold macro F1 (primary), weighted F1 and accuracy; per-class precision, recall and F1 **beside their support**; the confusion matrix with the semantic class names; and the out-of-fold `predict_proba` for P2.5.
- **Challenge B:** per-fold MAE(z) (primary), RMSE(z) and $R^2$(z), all computed directly from $z$ and $\hat z$, plus **one** MAE in nanoseconds computed only after inverting the prediction, $\widehat{\text{latency\_ns}} = 10^{\hat z}$, against the raw `latency_ns`.

In [11]:
import json
import time

from sklearn.base import clone
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             mean_absolute_error, mean_squared_error,
                             precision_recall_fscore_support, r2_score)

# Map class codes to their semantic names (used to label tables and the confusion matrix)
with open("../datos/cyber/cyber_class_codes.json") as f:
    CLASS_NAMES = {int(code): name for code, name in json.load(f).items()}
CLASS_CODES = sorted(CLASS_NAMES)


def evaluate_cyber(estimator, X, y, cv):
    """Challenge A under the metric contract, on the folds given by `cv`.

    The estimator is cloned and refitted on every fold, so any scaler or encoder inside
    a Pipeline is fitted on that fold's training portion only.

    Returns a dict with:
      fold_metrics : per-fold macro F1 (primary), weighted F1, accuracy, fit time
      per_class    : precision, recall, F1 and support of every class (out-of-fold, pooled)
      confusion    : confusion matrix (out-of-fold, pooled), labelled with class names
      proba        : out-of-fold predict_proba, if the estimator has it (else None)
    """
    y = np.asarray(y)
    # Out-of-fold containers: every row is predicted once, by a model that did not see it
    oof_pred = np.empty(len(y), dtype=y.dtype)
    oof_proba = np.full((len(y), len(CLASS_CODES)), np.nan)
    rows = []
    for k, (tr, va) in enumerate(cv.split(X, y)):
        # Fresh, unfitted copy of the estimator for this fold; time only the fit
        model = clone(estimator)
        t0 = time.perf_counter()
        model.fit(X.iloc[tr], y[tr])
        fit_time = time.perf_counter() - t0
        # Predict the validation part and store hard labels and class probabilities
        pred = model.predict(X.iloc[va])
        oof_pred[va] = pred
        if hasattr(model, "predict_proba"):
            cols = [CLASS_CODES.index(c) for c in model.classes_]  # align columns to class codes
            oof_proba[np.ix_(va, cols)] = model.predict_proba(X.iloc[va])
        # Per-fold metrics (macro F1 is the course-common primary metric)
        rows.append({"fold": k + 1,
                     "macro_f1": f1_score(y[va], pred, average="macro"),
                     "weighted_f1": f1_score(y[va], pred, average="weighted"),
                     "accuracy": accuracy_score(y[va], pred),
                     "fit_time_s": fit_time})

    # Per-class precision / recall / F1 beside the support, on the pooled out-of-fold predictions
    p, r, f, s = precision_recall_fscore_support(y, oof_pred, labels=CLASS_CODES, zero_division=0)
    per_class = pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s},
                             index=[CLASS_NAMES[c] for c in CLASS_CODES])
    # Confusion matrix (rows = true class, columns = predicted class) with semantic names
    names = [CLASS_NAMES[c] for c in CLASS_CODES]
    confusion = pd.DataFrame(confusion_matrix(y, oof_pred, labels=CLASS_CODES),
                             index=pd.Index(names, name="true"),
                             columns=pd.Index(names, name="predicted"))
    return {"fold_metrics": pd.DataFrame(rows).set_index("fold"), "per_class": per_class,
            "confusion": confusion,
            "proba": oof_proba if not np.isnan(oof_proba).all() else None}


def evaluate_storage(estimator, X, z, latency_ns, cv):
    """Challenge B under the metric contract and the log-to-physical rule, on `cv`.

    MAE(z), RMSE(z) and R2(z) are computed directly from z and z_hat. MAE_ns is computed
    only after inverting the prediction (10**z_hat) and against the raw latency_ns;
    10**MAE(z) is never reported as an error in nanoseconds.
    """
    z = np.asarray(z)
    latency_ns = np.asarray(latency_ns)
    rows = []
    for k, (tr, va) in enumerate(cv.split(X, z)):
        # Fresh, unfitted copy of the estimator for this fold; time only the fit
        model = clone(estimator)
        t0 = time.perf_counter()
        model.fit(X.iloc[tr], z[tr])
        fit_time = time.perf_counter() - t0
        z_hat = model.predict(X.iloc[va])
        # Log-scale metrics use z and z_hat directly; the nanosecond MAE inverts the
        # prediction first (10**z_hat) and compares it with the raw latency
        rows.append({"fold": k + 1,
                     "mae_z": mean_absolute_error(z[va], z_hat),
                     "rmse_z": np.sqrt(mean_squared_error(z[va], z_hat)),
                     "r2_z": r2_score(z[va], z_hat),
                     "mae_ns": mean_absolute_error(latency_ns[va], 10 ** z_hat),
                     "fit_time_s": fit_time})
    return {"fold_metrics": pd.DataFrame(rows).set_index("fold")}

## 10. Comparison table (`comparativa.csv`)

Template in `plantillas/comparativa.csv`. One row per (challenge, family, milestone, route); the table is **appended, never rewritten**.

In [12]:
from pathlib import Path

from sklearn.dummy import DummyClassifier, DummyRegressor

# Paths of the course template (gives the column layout) and of our comparison table
TEMPLATE_PATH = Path("../plantillas/comparativa.csv")
COMPARATIVA_PATH = Path("../comparativa.csv")
KEY = ["milestone", "challenge", "family", "route"]   # identifies one row of the table


def append_comparativa_row(row):
    """Append one row to comparativa.csv; the table is never rewritten.

    A new file gets only the template's header. A row whose (milestone, challenge, family,
    route) already exists is refused, so re-running the notebook cannot duplicate it.
    """
    columns = list(pd.read_csv(TEMPLATE_PATH).columns)
    assert set(row) == set(columns), f"row columns differ from the template: {set(row) ^ set(columns)}"
    if not COMPARATIVA_PATH.exists():
        pd.DataFrame(columns=columns).to_csv(COMPARATIVA_PATH, index=False)
    existing = pd.read_csv(COMPARATIVA_PATH)
    if ((existing[KEY] == pd.Series({k: row[k] for k in KEY})).all(axis=1)).any():
        print("already in comparativa.csv:", [row[k] for k in KEY])
        return
    pd.DataFrame([row], columns=columns).to_csv(COMPARATIVA_PATH, mode="a", header=False, index=False)
    print("appended:", [row[k] for k in KEY])


def fmt(values):
    """Format the per-fold values as one space-separated string with 4 decimals."""
    return " ".join(f"{v:.4f}" for v in values)


# Baseline rows, produced with the common evaluation functions on the canonical folds
latency_ns = storage["latency_ns"]
dummy_a = evaluate_cyber(DummyClassifier(strategy="most_frequent"), X_cyber, y_cyber, cv_cyber_canonical)
dummy_b = evaluate_storage(DummyRegressor(strategy="mean"), X_storage, z_storage, latency_ns,
                           cv_storage_canonical)

# Per-fold metrics of the two baselines
fa, fb = dummy_a["fold_metrics"], dummy_b["fold_metrics"]
display(fa.round(4))
display(fb.round(4))

# One comparison-table row per baseline, filled in the layout of the course template
row_cyber = {
    "milestone": "P2", "challenge": "cyber", "family": "Dummy", "route": "primary",
    "configuration": "strategy=most_frequent",
    "preprocessing": "none (the baseline ignores X)",
    "features": "features.md: the 11 distributed source predictors (frozen basis)",
    "validation": "canonical StratifiedGroupKFold(5) on feature_group, read from cyber_folds.csv",
    "primary_metric_per_fold": fmt(fa["macro_f1"]),
    "primary_metric": round(fa["macro_f1"].mean(), 4),
    "primary_metric_spread": round(fa["macro_f1"].max() - fa["macro_f1"].min(), 4),
    "supporting_metrics": f"weighted F1 {fa['weighted_f1'].mean():.4f}; accuracy {fa['accuracy'].mean():.4f}",
    "fit_time_s": round(fa["fit_time_s"].mean(), 4),
    "interpretability": "none",
    "notes": "predicts the majority class for every row: macro F1 is low because nine of ten classes get F1 = 0",
}
row_storage = {
    "milestone": "P2", "challenge": "storage", "family": "Dummy", "route": "primary",
    "configuration": "strategy=mean",
    "preprocessing": "none (the baseline ignores X); target z = log10(latency_ns)",
    "features": "features.md: the 9 distributed source predictors (frozen basis)",
    "validation": "canonical GroupKFold(5) on config_group, read from storage_folds.csv",
    "primary_metric_per_fold": fmt(fb["mae_z"]),
    "primary_metric": round(fb["mae_z"].mean(), 4),
    "primary_metric_spread": round(fb["mae_z"].max() - fb["mae_z"].min(), 4),
    "supporting_metrics": (f"RMSE(z) {fb['rmse_z'].mean():.4f}; R2(z) {fb['r2_z'].mean():.4f}; "
                           f"MAE_ns {fb['mae_ns'].mean():.0f} (computed after inverting predictions)"),
    "fit_time_s": round(fb["fit_time_s"].mean(), 4),
    "interpretability": "none",
    "notes": "constant prediction equal to the training mean of z; the reference every other family must beat",
}

# Append the rows (skipped if they are already in the table)
append_comparativa_row(row_cyber)
append_comparativa_row(row_storage)

,macro_f1,weighted_f1,accuracy,fit_time_s
fold,,,,
1,0.0242,0.0332,0.1374,0.0033
2,0.0242,0.0332,0.1374,0.0025
3,0.0242,0.0332,0.1374,0.0026
4,0.0242,0.0332,0.1374,0.0025
5,0.0242,0.0332,0.1374,0.0027


,mae_z,rmse_z,r2_z,mae_ns,fit_time_s
fold,,,,,
1,0.6800,0.8445,-0.0002,2.839968e+07,0.0014
2,0.6778,0.8436,-0.0002,2.854142e+07,0.0009
3,0.6628,0.8331,-0.0001,2.742680e+07,0.0009
4,0.6722,0.8269,-0.0001,2.407004e+07,0.0009
5,0.6922,0.8517,-0.0000,2.674962e+07,0.0014


already in comparativa.csv: ['P2', 'cyber', 'Dummy', 'primary']
already in comparativa.csv: ['P2', 'storage', 'Dummy', 'primary']
